In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

### Loading files

In [ ]:
arc = pd.read_csv("../data/raw/data_arc.csv")
bulk_additions = pd.read_csv("../data/raw/data_bulk.csv")
bulk_additions_time = pd.read_csv("../data/raw/data_bulk_time.csv")
stirring_gas = pd.read_csv("../data/raw/data_gas.csv")
temperature = pd.read_csv("../data/raw/data_temp.csv")
temperature_full = pd.read_csv("../data/raw/data_temp_FULL_with_test.csv")
wire_additions = pd.read_csv("../data/raw/data_wire.csv")
wire_additions_time = pd.read_csv("../data/raw/data_wire_time.csv")

datasets = {
    "arc": arc,
    "bulk_additions": bulk_additions,
    "bulk_additions_time": bulk_additions_time,
    "stirring_gas": stirring_gas,
    "temperature": temperature,
    "temperature_full": temperature_full,
    "wire_additions": wire_additions,
    "wire_additions_time": wire_additions_time,
}

In [ ]:
for name, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"  {name}  —  {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"{'='*60}")
    print(f"Columns: {list(df.columns)}")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nNull counts:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
    if df.isnull().sum().sum() == 0:
        print("No nulls.")
    print(f"\nSample rows:")
    display(df.head(3))

### Data types

Pandas inferred all timestamp columns as `object` (string) instead of `datetime64`. 
This affects the following files:

- `data_arc`: columns `Heating start` and `Heating end`
- `data_bulk_time`: columns `Bulk 1` through `Bulk 15`
- `data_wire_time`: columns `Wire 1` through `Wire 9`
- `data_temp` and `data_temp_full`: column `time`

All remaining columns were correctly inferred — numeric columns loaded as `int64` 
or `float64` as expected.

The timestamps follow a consistent format (`YYYY-MM-DD HH:MM:SS`) across all files, 
so conversion is straightforward. Passing the format explicitly to `pd.to_datetime` 
avoids the overhead of automatic format inference, which matters when iterating 
over 15 bulk and 9 wire columns.

### Converting data types

In [ ]:
arc["Heating start"] = pd.to_datetime(arc["Heating start"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
arc["Heating end"] = pd.to_datetime(arc["Heating end"], format="%Y-%m-%d %H:%M:%S", errors="coerce")

bulk_additions_time_cols = [f"Bulk {i}" for i in range(1, 16)]
for col in bulk_additions_time_cols:
    bulk_additions_time[col] = pd.to_datetime(bulk_additions_time[col], format="%Y-%m-%d %H:%M:%S", errors="coerce")

temperature["time"] = pd.to_datetime(temperature["time"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
temperature_full["time"] = pd.to_datetime(temperature_full["time"], format="%Y-%m-%d %H:%M:%S", errors="coerce")

wire_additions_time_cols = [f"Wire {i}" for i in range(1, 10)]
for col in wire_additions_time_cols:
    wire_additions_time[col] = pd.to_datetime(wire_additions_time[col], format="%Y-%m-%d %H:%M:%S", errors="coerce")


### Melt coverage across files

In [ ]:
print("Unique melts per file:")
for name, df in datasets.items():
    print(f"  {name:15s}: {df['key'].nunique()} melts")

all_keys = set(temperature_full["key"].unique())
for name, df in datasets.items():
    if name in ("temperature", "temperature_full"):
        continue
    missing = all_keys - set(df["key"].unique())
    extra = set(df["key"].unique()) - all_keys
    if missing:
        print(f"\n  {name}: {len(missing)} melts in temperature_full but NOT in {name}")
    if extra:
        print(f"\n  {name}: {len(extra)} melts in {name} but NOT in temperature_full")

if not any(
    (all_keys - set(df["key"].unique())) or (set(df["key"].unique()) - all_keys)
    for name, df in datasets.items()
    if name not in ("temperature", "temperature_full")
):
    print("\nAll files share the same set of melt keys.")

### Melt consistency across files

Not all files share the same set of melt keys. Some gaps are physically 
plausible; others are more likely registration failures.

- `arc`: 2 melts missing — arc heating is practically mandatory in LF 
  operations. The logistics in a melt shop impose waiting time between secondary 
  refining and casting, causing temperature drop that almost always requires 
  compensation. Ladle lids can reduce heat loss but rarely eliminate the need 
  for heating entirely. These 2 cases are most likely registration failures 
  rather than genuine absence of arc heating.

- `bulk_additions`: 87 melts missing — plausible for simple steel grades 
  where primary refining already delivers composition within specification. 
  The low count relative to total melts (2.7%) suggests these are genuine 
  exceptions rather than a data quality issue. These melts will be assigned 
  zero bulk addition features during feature engineering.

- `wire_additions`: 135 melts missing — wire treatment (specially CaSi for 
  inclusion modification) is a very common practice in 
  LF operations, particularly for continuous casting routes where inclusion 
  control is critical. 135 missing melts is high enough to warrant 
  investigation: if these keys are concentrated in a specific sequential 
  range, it may indicate an instrumentation failure during that period rather 
  than genuine absence of wire treatment.

- `stirring_gas`: 25 melts present in gas data but absent from temperature 
  records — bottom stirring is a mandatory operation. The absence of 
  temperature measurements for these melts suggests incomplete process cycles 
  or data collection failures. These 25 melts will be excluded from modeling 
  since there is no target variable to predict.

### Verifying temporal sequence of heats

In [ ]:
arc_timing = arc.groupby("key").agg(
    first_heat=("Heating start", "min"),
    last_heat=("Heating end", "max")
).reset_index().sort_values("first_heat")

print(arc_timing.head(20).to_string(index=False))

### Investigating the distribution of heats that miss wire information

In [ ]:
missing_wire_keys = sorted(all_keys - set(wire_additions["key"].unique()))

arc_missing = arc[arc["key"].isin(missing_wire_keys)][["key", "Heating start"]].sort_values("Heating start")

arc_missing["time_gap"] = arc_missing["Heating start"].diff()
block_threshold = pd.Timedelta(hours=2)
arc_missing["new_block"] = arc_missing["time_gap"] > block_threshold
arc_missing["block_id"] = arc_missing["new_block"].cumsum()

blocks = arc_missing.groupby("block_id").agg(
    n_heats=("key", "count"),
    start=("Heating start", "min"),
    end=("Heating start", "max")
).reset_index(drop=True)

print(f"Missing wire keys: {len(missing_wire_keys)}")
print(f"Number of temporal blocks: {len(blocks)}")
print(f"\nBlocks with 3 or more consecutive heats:")
print(blocks[blocks["n_heats"] >= 3].to_string(index=False))

### Wire addition gap investigation

To assess whether missing wire keys represent instrumentation failures or 
genuine process variation, the temporal sequence of heats was first verified 
using arc heating timestamps. The data confirms that `key` is assigned in 
strict chronological order — consecutive keys correspond to consecutive heats 
in time, with inter-heat gaps of approximately 5–10 minutes, consistent with 
ladle transfer time between primary refining and the LF station.

With chronological ordering confirmed, missing wire keys were grouped into 
temporal blocks — sequences of consecutive heats separated by less than 2 
hours. The analysis revealed 43 distinct blocks, of which 35 contain 3 or 
more consecutive heats without wire registration.

The largest blocks span 28, 24, 23 and 22 consecutive heats, covering periods 
of up to 4 hours of uninterrupted production. This pattern repeats consistently 
throughout the entire dataset (May to August 2019), across different days and 
hours, with no concentration in a specific period.

This rules out isolated instrumentation failure as the primary explanation. 
A more plausible interpretation is that this plant produces a mix of steel 
grades, a portion of which does not require wire treatment by specification. 
Simple structural steels, for example, often do not require inclusion 
modification with CaSi wire. Production scheduling typically groups similar 
grades in sequential campaigns, which would naturally produce the block 
structure observed here.

Implication for modeling: the 135 melts without wire records will be assigned 
zero wire addition features during feature engineering. Unlike the isolated 
gaps discussed earlier, these zeros may genuinely reflect absence of wire 
treatment rather than missing data — which means the model should be able to 
learn from them meaningfully. However, without grade information in the dataset, 
this cannot be confirmed, and the uncertainty should be kept in mind when 
interpreting wire-related features in the explainability notebook.

### Verifying temperature distribution and missingness

In [ ]:
print(f"temperature rows:      {len(temperature)}")
print(f"temperature_full rows: {len(temperature_full)}")

missing_in_temp = temperature["Temperature"].isna().sum()
print(f"\nMissing in temperature:      {missing_in_temp}")
print(f"Missing in temperature_full: {temperature_full['Temperature'].isna().sum()}")

extra_rows = len(temperature_full) - len(temperature)
print(f"\nExtra rows in temperature_full vs temperature: {extra_rows}")

In [ ]:
test_mask = temperature["Temperature"].isna()
print(f"Test rows (NaN in temperature): {test_mask.sum()}")
print(f"These rows in temperature_full — all have values: "
      f"{temperature_full.loc[test_mask.values, 'Temperature'].notna().all()}")

print(f"\nSample of test rows with ground truth:")
sample = temperature[test_mask].copy()
sample["true_temperature"] = temperature_full.loc[test_mask.values, "Temperature"].values
print(sample.head(10).to_string(index=False))

### Train/test structure

The dataset provides two temperature files with identical structure and row count 
(15907 rows each), but serving distinct roles:

- `data_temp.csv` contains 2901 missing temperature values — these are the rows 
  to predict, forming the test set.
- `data_temp_FULL_with_test.csv` contains all temperatures filled in, including 
  the 2901 test rows, and serves as the ground truth for final evaluation.

The test rows are not concentrated in specific melts or time periods — they are 
distributed across the dataset, which means the model will need to generalize 
across different operating conditions rather than memorize a specific subset.

Since the ground truth is available, model performance on the test set can be 
evaluated with real metrics after training. The ground truth file will not be 
used during training or validation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

temperature_full["Temperature"].dropna().hist(bins=50, ax=axes[0], edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Temperature (°C)")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of Measured Temperatures")

measurements_per_melt = temperature_full.groupby("key")["Temperature"].agg(
    total="count", missing=lambda x: x.isna().sum(), measured=lambda x: x.notna().sum()
)
measurements_per_melt["total"].hist(bins=30, ax=axes[1], edgecolor="black", alpha=0.7)
axes[1].set_xlabel("Number of Measurements")
axes[1].set_ylabel("Number of Melts")
axes[1].set_title("Measurements per Melt")

plt.tight_layout()
plt.show()

In [ ]:
print("Measurements per melt — summary statistics:")
display(measurements_per_melt.describe())

print(f"\nMelts with at least one missing temperature: "
      f"{(measurements_per_melt['missing'] > 0).sum()} / {len(measurements_per_melt)}")

In [ ]:
print("Temperatures below 1500°C:")
print(temperature_full[temperature_full["Temperature"] < 1500].shape[0])

print("\nTemperatures above 1680°C:")
print(temperature_full[temperature_full["Temperature"] > 1680].shape[0])

In [ ]:
cold = temperature_full[temperature_full["Temperature"] < 1500][["key", "time", "Temperature"]]
print(cold.to_string(index=False))

In [ ]:
for key in cold["key"].unique():
    melt_temps = temperature_full[temperature_full["key"] == key][["time", "Temperature"]].sort_values("time")
    print(f"\nMelt {key}:")
    print(melt_temps.to_string(index=False))

### Extreme temperature values

Eight temperature measurements fall below 1500°C, ranging from 1189°C to 1383°C. 
Since the melting point of carbon steel is approximately 1480–1520°C depending on 
composition, these values represent physically impossible readings for liquid steel 
in active LF operation.

Examining each case in context reveals a consistent pattern:

- Six measurements (melts 867, 1214, 1619, 2052, 2561 and 3034) show an abrupt 
  drop to implausible temperatures followed by an immediate return to normal values 
  within seconds to minutes. In melt 1214, a reading of 1208°C is followed 45 
  seconds later by 1608°C — a 400°C jump that is impossible for 
  a ladle of liquid steel.

- Melt 1818 shows two consecutive readings of 1383°C embedded between values above 
  1650°C, with less than 2 minutes separating all three points.

- Five of the eight anomalous readings occur as the first measurement of their 
  respective melts, suggesting the pyrometer had not yet stabilized or was not 
  properly immersed in the bath at the time of reading.

These are instrument errors, not real process events. They will be removed before 
feature engineering. Retaining them would corrupt the `prev_temp` feature for all 
subsequent measurements within the affected melts — a single bad reading would 
propagate incorrect thermal context through the entire melt's feature vector.

The 28 measurements above 1680°C were also reviewed. Unlike the cold readings, 
high temperatures in this range are physically plausible — intentional superheating 
occurs when a heat is expected to wait longer than usual before casting, for example 
when the tundish is still occupied. 
The operator heats above the normal target to compensate for the additional thermal 
loss during the extended waiting period. Peak readings during these heating cycles 
can reach 1680°C or above transiently.